In [ ]:


!pip install -q gdown h5py scipy pandas numpy matplotlib openpyxl

import os
import re
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import gdown

from google.colab import files
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import pearsonr, spearmanr


OUTPUT_DIR = "endpoint_calibrated_phasefit_PL_KPFM_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIME_STEP_MIN = 9

PL_READ_FOR_ANALYSIS = "top"

PRIMARY_PL_TIMEPOINT_MODE = "initial"
MAKE_LAST_VALID_SENSITIVITY_FIGURE = True

PL_PLOT_WL_MIN = 450
PL_PLOT_WL_MAX = 850

PL_FIT_WL_MIN = 500
PL_FIT_WL_MAX = 835

KPFM_GDRIVE_URL = ""
H5_PATH = "photokpfm_measurements.h5"

USE_ABS_PHOTOVOLTAGE = True


MAKE_KPFM_DELTA_LIGHT_DARK_FIGURES = True


FORCE_ALL_FA_FREE_WELLS_TO_N1 = True


FORCE_OVERSHOOT_PL_TO_N1 = True
ZERO_FA_ALLOWED_PHASES = ["n=1-like / 2D"]
OVERSHOOT_LOW_N_WINDOW = (500, 585)
OVERSHOOT_HIGH_FRAC = 0.88
OVERSHOOT_MIN_SPAN_NM = 18.0

SORT_MODE = "FA_then_BDA"


MAKE_KPFM_DISTRIBUTION_PLOTS = True
MAKE_ALL_KPFM_DISTRIBUTION_PLOTS = True

KPFM_DISTRIBUTION_CROP = 10
KPFM_DISTRIBUTION_BINS = 50
KPFM_DISTRIBUTION_HIST_RANGE = (-2.5, -0.5)




PHASE_WINDOWS = {
    "n=1-like / 2D":   (500, 585),
    "n=2-like":        (585, 625),
    "n=3-like":        (625, 670),
    "n=4-like":        (670, 715),
    "n>=5 / high-n":   (715, 760),
    "3D / 3D-like":    (760, 835),
}

PHASE_ORDER = [
    "n=1-like / 2D",
    "n=2-like",
    "n=3-like",
    "n=4-like",
    "n>=5 / high-n",
    "3D / 3D-like",
]

PHASE_COLORS = {
    "n=1-like / 2D": "#5DA5DA",
    "n=2-like": "#60BD68",
    "n=3-like": "#F17CB0",
    "n=4-like": "#B2912F",
    "n>=5 / high-n": "#B276B2",
    "3D / 3D-like": "#F15854",
    "Unassigned": "#B0B0B0",
}


SMOOTH_WINDOW = 11
SMOOTH_POLYORDER = 2

MIN_PHASE_HEIGHT_FRAC = 0.020
MIN_AREA_FRAC_TO_KEEP = 0.035
MIN_AMP_FRAC_TO_KEEP = 0.015

ENABLE_RESIDUAL_SECOND_PASS = True
MIN_RESIDUAL_HEIGHT_FRAC = 0.035
MIN_RESIDUAL_AREA_FRAC = 0.030

SIGMA_MIN = 4.0
SIGMA_MAX_LOW_N = 35.0
SIGMA_MAX_HIGH_N = 60.0
SIGMA_MAX_3D = 50.0

EDGE_MARGIN_NM = 4.0
FORCE_ONE_COMPONENT_IF_EMPTY = True

ROWS = list("ABCDEFGH")
COLS = list(range(1, 13))
WELLS_ROW_MAJOR = [f"{r}{c}" for r in ROWS for c in COLS]


print("Upload your PL plate-reader CSV file now.")
uploaded = files.upload()

CSV_PATH = None
for fname in uploaded.keys():
    if fname.lower().endswith(".csv"):
        CSV_PATH = fname
        break

if CSV_PATH is None:
    raise FileNotFoundError("No CSV file was uploaded.")

print("Using PL CSV:", CSV_PATH)


if not os.path.exists(H5_PATH) and KPFM_GDRIVE_URL:
    print("Downloading photoKPFM H5 file...")
    gdown.download(KPFM_GDRIVE_URL, H5_PATH, quiet=False, fuzzy=True)

if not os.path.exists(H5_PATH):
    print("Could not download H5 automatically. Upload the H5 file manually now.")
    uploaded_h5 = files.upload()
    for fname in uploaded_h5.keys():
        if fname.lower().endswith(".h5") or fname.lower().endswith(".hdf5"):
            H5_PATH = fname
            break

if not os.path.exists(H5_PATH):
    raise FileNotFoundError("No H5 file found or uploaded.")

print("Using photoKPFM H5:", H5_PATH)


COMPOSITION_PERCENT_MAP = {
    "A1":  (100, 0, 0),   "A2":  (76, 8, 16),  "A3":  (60, 32, 8),  "A4":  (52, 24, 24),
    "A5":  (44, 24, 32),  "A6":  (36, 32, 32), "A7":  (28, 48, 24), "A8":  (20, 72, 8),
    "A9":  (20, 8, 72),   "A10": (12, 40, 48), "A11": (4, 80, 16),  "A12": (4, 16, 80),

    "B1":  (92, 8, 0),    "B2":  (76, 0, 24),  "B3":  (60, 24, 16), "B4":  (52, 16, 32),
    "B5":  (44, 16, 40),  "B6":  (36, 24, 40), "B7":  (28, 40, 32), "B8":  (20, 64, 16),
    "B9":  (20, 0, 80),   "B10": (12, 32, 56), "B11": (4, 72, 24),  "B12": (4, 8, 88),

    "C1":  (92, 0, 8),    "C2":  (68, 32, 0),  "C3":  (60, 16, 24), "C4":  (52, 8, 40),
    "C5":  (44, 8, 48),   "C6":  (36, 16, 48), "C7":  (28, 32, 40), "C8":  (20, 56, 24),
    "C9":  (12, 8, 80),   "C10": (12, 24, 64), "C11": (4, 64, 32),  "C12": (4, 0, 96),

    "D1":  (84, 16, 0),   "D2":  (68, 24, 8),  "D3":  (60, 8, 32),  "D4":  (52, 0, 48),
    "D5":  (44, 0, 56),   "D6":  (36, 8, 56),  "D7":  (28, 24, 48), "D8":  (20, 48, 32),
    "D9":  (12, 0, 88),   "D10": (12, 16, 72), "D11": (4, 56, 40),  "D12": (4, 56, 40),

    "E1":  (84, 8, 8),    "E2":  (68, 16, 16), "E3":  (60, 0, 40),  "E4":  (44, 56, 0),
    "E5":  (36, 64, 0),   "E6":  (36, 0, 64),  "E7":  (28, 16, 56), "E8":  (20, 40, 40),
    "E9":  (12, 72, 16),  "E10": (12, 8, 80),  "E11": (4, 48, 48),  "E12": (4, 48, 48),

    "F1":  (84, 0, 16),   "F2":  (68, 8, 24),  "F3":  (52, 48, 0),  "F4":  (44, 48, 8),
    "F5":  (36, 56, 8),   "F6":  (28, 72, 0),  "F7":  (28, 8, 64),  "F8":  (20, 32, 48),
    "F9":  (12, 64, 24),  "F10": (12, 0, 88),  "F11": (4, 40, 56),  "F12": (4, 40, 56),

    "G1":  (76, 24, 0),   "G2":  (68, 0, 32),  "G3":  (52, 40, 8),  "G4":  (44, 40, 16),
    "G5":  (36, 48, 16),  "G6":  (28, 64, 8),  "G7":  (28, 0, 72),  "G8":  (20, 24, 56),
    "G9":  (12, 56, 32),  "G10": (4, 96, 0),   "G11": (4, 32, 64),  "G12": (4, 32, 64),

    "H1":  (76, 16, 8),   "H2":  (60, 40, 0),  "H3":  (52, 32, 16), "H4":  (44, 32, 24),
    "H5":  (36, 40, 24),  "H6":  (28, 56, 16), "H7":  (20, 80, 0),  "H8":  (20, 16, 64),
    "H9":  (12, 48, 40),  "H10": (4, 8, 88),   "H11": (4, 24, 72),  "H12": (4, 24, 72),
}


def safe_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    if s.upper() in {"OVRFLW", "OVERFLOW", "OVER"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def normalize_well_name(x):
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    m = re.match(r"^([A-H])0?([1-9]|1[0-2])$", s)
    if m:
        return f"{m.group(1)}{int(m.group(2))}"
    return None


def well_sort_key(well):
    return (ROWS.index(well[0]), int(well[1:]))


def kpfm_index_to_well(idx):
    idx = int(idx)
    col = idx // 8 + 1
    row = ROWS[idx % 8]
    return f"{row}{col}"


def well_to_kpfm_index(well):
    row_idx = ROWS.index(well[0])
    col_idx = int(well[1:]) - 1
    return col_idx * 8 + row_idx


def make_composition_df():
    rows_out = []
    for well, vals in COMPOSITION_PERCENT_MAP.items():
        pea, bda, fa = vals
        total = pea + bda + fa
        rows_out.append({
            "Well": well,
            "KPFM_index": well_to_kpfm_index(well),
            "PEA_pct": pea,
            "BDA_pct": bda,
            "FA_pct": fa,
            "PEA_frac": pea / total if total > 0 else np.nan,
            "BDA_frac": bda / total if total > 0 else np.nan,
            "FA_frac": fa / total if total > 0 else np.nan,
        })
    df = pd.DataFrame(rows_out)
    return df.sort_values("Well", key=lambda s: [well_sort_key(x) for x in s]).reset_index(drop=True)


def make_composition_label(row):
    return f"{int(row['PEA_pct'])}% PEA\n{int(row['BDA_pct'])}% BDA\n{int(row['FA_pct'])}% FA"


def sort_matched_df(df, mode="FA_then_BDA", pv_col=None):
    if mode == "FA_then_BDA":
        return df.sort_values(
            ["FA_pct", "BDA_pct", "PEA_pct", "Well"],
            ascending=[True, True, False, True],
        ).reset_index(drop=True)

    if mode == "BDA_then_FA":
        return df.sort_values(
            ["BDA_pct", "FA_pct", "PEA_pct", "Well"],
            ascending=[True, True, False, True],
        ).reset_index(drop=True)

    if mode == "plate_order":
        return df.sort_values(
            "Well",
            key=lambda s: [well_sort_key(x) for x in s],
        ).reset_index(drop=True)

    if mode == "KPFM_descending" and pv_col is not None:
        return df.sort_values(pv_col, ascending=False).reset_index(drop=True)

    return df.reset_index(drop=True)


def area_of_gaussian(A, sigma):
    return float(abs(A) * abs(sigma) * np.sqrt(2 * np.pi))


def gaussian1(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def multi_gaussian(x, *params):
    y = np.zeros_like(x, dtype=float)
    for i in range(0, len(params), 3):
        A, mu, sigma = params[i:i+3]
        y += gaussian1(x, A, mu, sigma)
    return y


def phase_sigma_max(phase):
    if phase in ["n=1-like / 2D", "n=2-like", "n=3-like", "n=4-like"]:
        return SIGMA_MAX_LOW_N
    if phase == "3D / 3D-like":
        return SIGMA_MAX_3D
    return SIGMA_MAX_HIGH_N


def phase_for_mu(mu):
    for phase, (lo, hi) in PHASE_WINDOWS.items():
        if lo <= mu < hi:
            return phase
    return "Unassigned"


def allowed_phases_for_well(well):
    """
    Composition-aware constraint for this PEA/BDA/FA library.

    For wells with no FA, there is no 3D FAPbI3-forming component in the
    precursor recipe. In the corrected setting, FA-free PEA/BDA endpoint wells
    are only allowed to contribute to n=1-like / 2D. This prevents the pure
    endpoint PL from being split into artificial n=2/high-n/3D fractions.
    """
    if well in COMPOSITION_PERCENT_MAP:
        pea, bda, fa = COMPOSITION_PERCENT_MAP[well]
        if fa == 0:
            return list(ZERO_FA_ALLOWED_PHASES)
    return list(PHASE_ORDER)


def is_overshoot_low_n_endpoint(well, wl_plot, y_corr_plot):
    """
    Detect broad clipped/overshot low-dimensional endpoint PL and force it to
    n=1-like / 2D.

    This is intentionally restricted to FA-free PEA/BDA endpoint wells so that
    real FA-containing high-n/3D emission is not suppressed.
    """
    if not FORCE_OVERSHOOT_PL_TO_N1:
        return False

    if well not in COMPOSITION_PERCENT_MAP:
        return False

    pea, bda, fa = COMPOSITION_PERCENT_MAP[well]

    if fa != 0:
        return False

    wl = np.asarray(wl_plot, dtype=float)
    y = np.asarray(y_corr_plot, dtype=float)

    good = np.isfinite(wl) & np.isfinite(y)

    if good.sum() < 10:
        return False

    wl = wl[good]
    y = y[good]

    ymax = float(np.nanmax(y))

    if ymax <= 0:
        return False

    lo, hi = OVERSHOOT_LOW_N_WINDOW
    low_mask = (wl >= lo) & (wl <= hi)

    if low_mask.sum() < 5:
        return False

    low_y = y[low_mask]
    low_wl = wl[low_mask]


    peak_wl = float(wl[int(np.nanargmax(y))])

    if not (lo <= peak_wl <= hi):
        return False

    high_mask = low_y >= (OVERSHOOT_HIGH_FRAC * ymax)

    if high_mask.sum() < 2:
        return False

    high_span = float(np.nanmax(low_wl[high_mask]) - np.nanmin(low_wl[high_mask]))

    return high_span >= OVERSHOOT_MIN_SPAN_NM


def make_forced_n1_fit_result(wl_plot, y_corr_plot, y_smooth_plot, wl_fit, reason="forced_n1"):
    """
    Return a valid fit object with one n=1-like / 2D component.
    The exact Gaussian is only for plotting; the phase fraction is 100% n=1.
    """
    if len(wl_plot) == 0:
        return {
            "ok": False,
            "wl_plot": wl_plot,
            "y_corr_plot": y_corr_plot,
            "y_smooth_plot": y_smooth_plot,
            "wl_fit": wl_fit,
            "fit_y_plot": np.full_like(wl_plot, np.nan),
            "peaks": [],
        }

    lo, hi = PHASE_WINDOWS["n=1-like / 2D"]
    mask = (wl_plot >= lo) & (wl_plot <= hi) & np.isfinite(y_corr_plot)

    if mask.sum() >= 3 and np.nanmax(y_corr_plot[mask]) > 0:
        idx_local = int(np.nanargmax(y_corr_plot[mask]))
        mu0 = float(wl_plot[mask][idx_local])
        A0 = float(np.nanmax(y_corr_plot[mask]))
    else:
        good = np.isfinite(y_corr_plot)
        idx = int(np.nanargmax(y_corr_plot[good])) if good.sum() else 0
        mu0 = float(wl_plot[good][idx]) if good.sum() else 555.0
        A0 = float(np.nanmax(y_corr_plot[good])) if good.sum() else 1.0

    mu0 = min(max(mu0, lo + 1.0), hi - 1.0)
    sigma0 = 18.0
    area0 = area_of_gaussian(A0, sigma0)

    peaks = [{
        "phase": "n=1-like / 2D",
        "source": reason,
        "A": A0,
        "mu_nm": mu0,
        "sigma_nm": sigma0,
        "area": area0,
    }]

    fit_y_plot = gaussian1(wl_plot, A0, mu0, sigma0)

    return {
        "ok": True,
        "wl_plot": wl_plot,
        "y_corr_plot": y_corr_plot,
        "y_smooth_plot": y_smooth_plot,
        "wl_fit": wl_fit,
        "fit_y_plot": fit_y_plot,
        "peaks": peaks,
    }


def normalize_overlay_to_0_1(values, mode="auto"):
    """
    Normalize an overlay for plotting on top of the stacked phase distribution.

    - For positive-only quantities like |light-dark|, divide by max.
    - For signed light/dark potentials, use min-max normalization so negative
      surface-potential values are still visually comparable.
    """
    s = pd.to_numeric(pd.Series(values), errors="coerce")
    arr = s.to_numpy(dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    good = np.isfinite(arr)

    if good.sum() == 0:
        return out

    vals = arr[good]

    if mode == "max" or (mode == "auto" and np.nanmin(vals) >= 0):
        vmax = float(np.nanmax(vals))
        if vmax != 0:
            out[good] = vals / vmax
        return out

    vmin = float(np.nanmin(vals))
    vmax = float(np.nanmax(vals))

    if vmax > vmin:
        out[good] = (vals - vmin) / (vmax - vmin)
    else:
        out[good] = 0.5

    return out


def parse_plate_reader_blocks(csv_path):
    raw = pd.read_csv(csv_path, header=None, dtype=str)

    starts = []
    for i in range(len(raw)):
        row_text = " ".join([str(x) for x in raw.iloc[i].tolist() if not pd.isna(x)])
        if re.search(r"Read\s+\d+\s*:", row_text):
            starts.append(i)

    if len(starts) == 0:
        raise ValueError("No Read blocks found. Check CSV format.")

    parsed_blocks = []

    for i, start in enumerate(starts):
        end = starts[i + 1] if i + 1 < len(starts) else len(raw)

        header_row = None
        wavelength_col = None

        for r in range(start, min(end, start + 40)):
            for c in range(raw.shape[1]):
                val = str(raw.iat[r, c]).strip() if not pd.isna(raw.iat[r, c]) else ""
                if val.lower() == "wavelength":
                    header_row = r
                    wavelength_col = c
                    break
            if header_row is not None:
                break

        if header_row is None or wavelength_col is None:
            continue

        header_vals = raw.iloc[header_row].tolist()

        well_cols = []
        seen = set()

        for c in range(wavelength_col + 1, raw.shape[1]):
            well = normalize_well_name(header_vals[c])
            if well is not None and well in WELLS_ROW_MAJOR and well not in seen:
                well_cols.append((c, well))
                seen.add(well)

        print(f"Read block {len(parsed_blocks) + 1}: found {len(well_cols)} well columns")

        if len(well_cols) == 0:
            continue

        rows_block = []

        for r in range(header_row + 1, end):
            wl = safe_float(raw.iat[r, wavelength_col])
            if not np.isfinite(wl):
                continue

            row = {"Wavelength": wl}
            for c, well in well_cols:
                row[well] = safe_float(raw.iat[r, c])

            rows_block.append(row)

        if len(rows_block) == 0:
            continue

        df = pd.DataFrame(rows_block)

        for well in WELLS_ROW_MAJOR:
            if well not in df.columns:
                df[well] = np.nan

        df = df[["Wavelength"] + WELLS_ROW_MAJOR].copy()
        parsed_blocks.append(df)

    if len(parsed_blocks) == 0:
        raise ValueError("No usable spectral blocks were parsed.")

    top_blocks = []
    bottom_blocks = []
    abs_blocks = []

    for i, block in enumerate(parsed_blocks):
        if i % 3 == 0:
            top_blocks.append(block)
        elif i % 3 == 1:
            bottom_blocks.append(block)
        else:
            abs_blocks.append(block)

    n_cycles = min(len(top_blocks), len(bottom_blocks), len(abs_blocks))

    top_blocks = top_blocks[:n_cycles]
    bottom_blocks = bottom_blocks[:n_cycles]
    abs_blocks = abs_blocks[:n_cycles]

    print("Parsed blocks:", len(parsed_blocks))
    print("Top PL timepoints:", len(top_blocks))
    print("Bottom PL timepoints:", len(bottom_blocks))
    print("Absorbance timepoints:", len(abs_blocks))
    print("Full cycles:", n_cycles)

    return {"top": top_blocks, "bottom": bottom_blocks, "abs": abs_blocks}


def summarize_pl(blocks, read_name):
    rows_out = []

    for t_idx, df in enumerate(blocks, start=1):
        wl = pd.to_numeric(df["Wavelength"], errors="coerce").to_numpy()
        mask = np.isfinite(wl) & (wl >= PL_PLOT_WL_MIN) & (wl <= PL_PLOT_WL_MAX)
        wl_use = wl[mask]

        for well in WELLS_ROW_MAJOR:
            y = pd.to_numeric(df[well], errors="coerce").to_numpy()
            y_use = y[mask]

            good = np.isfinite(wl_use) & np.isfinite(y_use)

            if good.sum() < 5:
                peak_intensity = np.nan
                peak_wavelength = np.nan
                integrated_PL = np.nan
            else:
                wl_good = wl_use[good]
                y_good = y_use[good]
                peak_idx = int(np.nanargmax(y_good))

                peak_intensity = float(y_good[peak_idx])
                peak_wavelength = float(wl_good[peak_idx])
                integrated_PL = float(np.trapezoid(y_good, wl_good))

            rows_out.append({
                "Well": well,
                "Read": read_name,
                "Timepoint": t_idx,
                "Time_min": (t_idx - 1) * TIME_STEP_MIN,
                "peak_intensity": peak_intensity,
                "peak_wavelength_nm": peak_wavelength,
                "integrated_PL": integrated_PL,
            })

    return pd.DataFrame(rows_out)


def choose_timepoint(pl_all, mode, read_name):
    valid_counts = (
        pl_all[pl_all["Read"] == read_name]
        .groupby("Timepoint")["peak_intensity"]
        .apply(lambda s: int(s.notna().sum()))
        .reset_index(name="non_nan_PL_count")
    )

    print("\nPL valid counts by timepoint:")
    display(valid_counts)

    valid_timepoints = valid_counts[
        valid_counts["non_nan_PL_count"] > 0
    ]["Timepoint"].tolist()

    if len(valid_timepoints) == 0:
        raise ValueError("No usable PL timepoints found.")

    if mode == "initial":
        chosen = int(valid_timepoints[0])
        label = "initial"

    elif mode == "last_valid":
        chosen = int(valid_timepoints[-1])
        label = "last valid"

    elif isinstance(mode, int):
        if mode not in valid_timepoints:
            raise ValueError(f"Requested timepoint {mode} is not valid. Valid timepoints: {valid_timepoints}")
        chosen = int(mode)
        label = f"manual timepoint {chosen}"

    else:
        raise ValueError("Timepoint mode must be 'initial', 'last_valid', or an integer.")

    chosen_min = int((chosen - 1) * TIME_STEP_MIN)
    chosen_count = int(valid_counts[valid_counts["Timepoint"] == chosen]["non_nan_PL_count"].iloc[0])

    print(f"Using {label} PL timepoint:", chosen)
    print("Chosen PL time:", chosen_min, "min")
    print("Non-NaN PL values at chosen timepoint:", chosen_count)

    return chosen, chosen_min, label, valid_counts


def recursively_load_hdf5_group(group):
    out = {}
    for key, item in group.items():
        if isinstance(item, h5py.Dataset):
            data = item[()]
            if isinstance(data, bytes):
                data = data.decode("utf-8")
            out[key] = data
        elif isinstance(item, h5py.Group):
            nested = recursively_load_hdf5_group(item)
            for nk, nv in nested.items():
                out[nk] = nv
    return out


def load_hdf5_to_dict(file_path):
    with h5py.File(file_path, "r") as f:
        return recursively_load_hdf5_group(f)


def recalculate_y_train_from_hist(res_dict, crop=10, bins=30, hist_range=(-2.5, -0.5), fallback="mean"):
    dark_data = np.asarray(res_dict["dark_data"])
    light_data = np.asarray(res_dict["light_data"])

    n = len(dark_data)

    y_new = np.full(n, np.nan)
    mu_dark_all = np.full(n, np.nan)
    mu_light_all = np.full(n, np.nan)

    for i in range(n):
        dark_img = np.asarray(dark_data[i], dtype=float)
        light_img = np.asarray(light_data[i], dtype=float)

        if crop > 0 and dark_img.ndim >= 2 and light_img.ndim >= 2:
            dark_vals = dark_img[crop:-crop, crop:-crop].ravel()
            light_vals = light_img[crop:-crop, crop:-crop].ravel()
        else:
            dark_vals = dark_img.ravel()
            light_vals = light_img.ravel()

        dark_vals = dark_vals[np.isfinite(dark_vals)]
        light_vals = light_vals[np.isfinite(light_vals)]

        if len(dark_vals) < 20 or len(light_vals) < 20:
            continue

        counts_d, bin_edges_d = np.histogram(dark_vals, bins=bins, range=hist_range)
        bin_centers_d = 0.5 * (bin_edges_d[:-1] + bin_edges_d[1:])

        counts_l, bin_edges_l = np.histogram(light_vals, bins=bins, range=hist_range)
        bin_centers_l = 0.5 * (bin_edges_l[:-1] + bin_edges_l[1:])

        try:
            p0_d = [
                max(float(counts_d.max()), 1.0),
                float(np.mean(dark_vals)),
                max(float(np.std(dark_vals)), 1e-4),
            ]
            popt_d, _ = curve_fit(gaussian1, bin_centers_d, counts_d, p0=p0_d, maxfev=5000)
            mu_d = popt_d[1]

            p0_l = [
                max(float(counts_l.max()), 1.0),
                float(np.mean(light_vals)),
                max(float(np.std(light_vals)), 1e-4),
            ]
            popt_l, _ = curve_fit(gaussian1, bin_centers_l, counts_l, p0=p0_l, maxfev=5000)
            mu_l = popt_l[1]

        except Exception:
            if fallback == "mean":
                mu_d = np.mean(dark_vals)
                mu_l = np.mean(light_vals)
            elif fallback == "median":
                mu_d = np.median(dark_vals)
                mu_l = np.median(light_vals)
            else:
                continue

        y_new[i] = mu_l - mu_d
        mu_dark_all[i] = mu_d
        mu_light_all[i] = mu_l

    return y_new, mu_dark_all, mu_light_all


def summarize_kpfm(h5_path):
    res_dict = load_hdf5_to_dict(h5_path)

    print("photoKPFM H5 keys:")
    for k in res_dict.keys():
        try:
            print(" ", k, np.shape(res_dict[k]))
        except Exception:
            print(" ", k, type(res_dict[k]))

    if "idx" not in res_dict:
        raise ValueError("H5 file does not contain idx. Cannot map KPFM measurements to wells.")

    measured_idx = np.ravel(res_dict["idx"]).astype(int)

    if "dark_data" in res_dict and "light_data" in res_dict:
        y_signed, mu_dark, mu_light = recalculate_y_train_from_hist(
            res_dict,
            crop=10,
            bins=30,
            hist_range=(-2.5, -0.5),
            fallback="mean",
        )

    elif "y_train" in res_dict:
        y_signed = np.ravel(res_dict["y_train"]).astype(float)
        mu_dark = np.full_like(y_signed, np.nan, dtype=float)
        mu_light = np.full_like(y_signed, np.nan, dtype=float)

    else:
        raise ValueError("Could not find dark_data/light_data or y_train in H5.")

    n = min(len(measured_idx), len(y_signed))

    rows_out = []

    for j in range(n):
        plate_idx = int(measured_idx[j])
        well = kpfm_index_to_well(plate_idx)
        val = float(y_signed[j]) if np.isfinite(y_signed[j]) else np.nan

        rows_out.append({
            "KPFM_index": plate_idx,
            "KPFM_measurement_number": j,
            "Well": well,
            "photovoltage_signed_V": val,
            "photovoltage_abs_V": abs(val) if np.isfinite(val) else np.nan,
            "mu_dark_V": mu_dark[j] if j < len(mu_dark) else np.nan,
            "mu_light_V": mu_light[j] if j < len(mu_light) else np.nan,
        })

    out = pd.DataFrame(rows_out)

    print("\nMeasured KPFM point mapping:")
    display(out[[
        "KPFM_measurement_number",
        "KPFM_index",
        "Well",
        "photovoltage_signed_V",
        "photovoltage_abs_V",
    ]])

    return out


def fit_gaussian_to_hist(vals, bins=50, hist_range=(-2.5, -0.5)):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) < 20:
        return None

    counts, edges = np.histogram(vals, bins=bins, range=hist_range)
    centers = 0.5 * (edges[:-1] + edges[1:])

    if counts.max() <= 0:
        return None

    p0 = [
        float(counts.max()),
        float(np.mean(vals)),
        max(float(np.std(vals)), 1e-4),
    ]

    try:
        popt, _ = curve_fit(
            gaussian1,
            centers,
            counts,
            p0=p0,
            maxfev=10000,
        )
    except Exception:
        popt = p0

    A, mu, sigma = popt

    return {
        "counts": counts,
        "edges": edges,
        "centers": centers,
        "A": float(A),
        "mu": float(mu),
        "sigma": float(abs(sigma)),
    }


def get_cropped_kpfm_values(img, crop=10):
    img = np.asarray(img, dtype=float)

    if crop > 0 and img.ndim >= 2 and img.shape[0] > 2 * crop and img.shape[1] > 2 * crop:
        vals = img[crop:-crop, crop:-crop].ravel()
    else:
        vals = img.ravel()

    vals = vals[np.isfinite(vals)]
    return vals


def plot_single_kpfm_distribution(
    dark_vals,
    light_vals,
    well,
    composition_row=None,
    bins=50,
    hist_range=(-2.5, -0.5),
    output_png=None,
    output_svg=None,
    show=True,
):
    dark_fit = fit_gaussian_to_hist(
        dark_vals,
        bins=bins,
        hist_range=hist_range,
    )

    light_fit = fit_gaussian_to_hist(
        light_vals,
        bins=bins,
        hist_range=hist_range,
    )

    if dark_fit is None or light_fit is None:
        print(f"Skipping {well}: not enough KPFM pixels for histogram fit.")
        return None

    sp_dark = dark_fit["mu"]
    sp_light = light_fit["mu"]
    delta_v = sp_light - sp_dark

    xfit = np.linspace(hist_range[0], hist_range[1], 600)

    fig, ax = plt.subplots(figsize=(5.0, 4.0), dpi=250)

    ax.hist(
        dark_vals,
        bins=bins,
        range=hist_range,
        alpha=0.65,
        label="dark pixels",
        edgecolor="black",
        linewidth=0.3,
    )

    ax.hist(
        light_vals,
        bins=bins,
        range=hist_range,
        alpha=0.65,
        label="light pixels",
        edgecolor="black",
        linewidth=0.3,
    )

    ax.plot(
        xfit,
        gaussian1(xfit, dark_fit["A"], dark_fit["mu"], dark_fit["sigma"]),
        "--",
        linewidth=2.0,
        label="dark Gaussian fit",
    )

    ax.plot(
        xfit,
        gaussian1(xfit, light_fit["A"], light_fit["mu"], light_fit["sigma"]),
        "--",
        linewidth=2.0,
        label="light Gaussian fit",
    )

    ax.axvline(sp_dark, linestyle=":", linewidth=1.2)
    ax.axvline(sp_light, linestyle=":", linewidth=1.2)

    if composition_row is not None:
        subtitle = (
            f"{well}: {int(composition_row['PEA_pct'])}% PEA / "
            f"{int(composition_row['BDA_pct'])}% BDA / "
            f"{int(composition_row['FA_pct'])}% FA"
        )
    else:
        subtitle = well

    ax.set_title(
        f"{subtitle}\n"
        f"$SP_{{dark}}$ = {sp_dark:.2f} V; "
        f"$SP_{{light}}$ = {sp_light:.2f} V; "
        f"$\\Delta V$ = {delta_v:.2f} V",
        fontsize=10,
    )

    ax.set_xlabel("Surface potential (V)")
    ax.set_ylabel("Pixel count")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7, loc="upper left")

    fig.tight_layout()

    if output_png is not None:
        fig.savefig(output_png, dpi=500, bbox_inches="tight")

    if output_svg is not None:
        fig.savefig(output_svg, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return {
        "Well": well,
        "SP_dark_fit_V": sp_dark,
        "SP_light_fit_V": sp_light,
        "deltaV_fit_V": delta_v,
        "dark_sigma_V": dark_fit["sigma"],
        "light_sigma_V": light_fit["sigma"],
        "dark_pixel_mean_V": float(np.mean(dark_vals)),
        "light_pixel_mean_V": float(np.mean(light_vals)),
        "dark_pixel_median_V": float(np.median(dark_vals)),
        "light_pixel_median_V": float(np.median(light_vals)),
        "dark_pixel_std_V": float(np.std(dark_vals)),
        "light_pixel_std_V": float(np.std(light_vals)),
        "n_dark_pixels": int(len(dark_vals)),
        "n_light_pixels": int(len(light_vals)),
    }


def make_all_kpfm_distribution_plots(
    h5_path,
    composition_df,
    output_dir,
    crop=10,
    bins=50,
    hist_range=(-2.5, -0.5),
):
    dist_dir = os.path.join(output_dir, "KPFM_light_dark_distribution_plots")
    os.makedirs(dist_dir, exist_ok=True)

    res_dict = load_hdf5_to_dict(h5_path)

    if "dark_data" not in res_dict or "light_data" not in res_dict or "idx" not in res_dict:
        print("KPFM distribution plots skipped: H5 does not contain dark_data, light_data, and idx.")
        return pd.DataFrame()

    dark_data = np.asarray(res_dict["dark_data"])
    light_data = np.asarray(res_dict["light_data"])
    measured_idx = np.ravel(res_dict["idx"]).astype(int)

    rows_out = []

    for j, plate_idx in enumerate(measured_idx):
        well = kpfm_index_to_well(int(plate_idx))

        comp_match = composition_df[composition_df["Well"] == well]
        composition_row = None if comp_match.empty else comp_match.iloc[0]

        dark_vals = get_cropped_kpfm_values(dark_data[j], crop=crop)
        light_vals = get_cropped_kpfm_values(light_data[j], crop=crop)

        output_png = os.path.join(dist_dir, f"{well}_KPFM_light_dark_distribution.png")
        output_svg = os.path.join(dist_dir, f"{well}_KPFM_light_dark_distribution.svg")

        result = plot_single_kpfm_distribution(
            dark_vals=dark_vals,
            light_vals=light_vals,
            well=well,
            composition_row=composition_row,
            bins=bins,
            hist_range=hist_range,
            output_png=output_png,
            output_svg=output_svg,
            show=False,
        )

        if result is not None:
            if composition_row is not None:
                result["PEA_pct"] = composition_row["PEA_pct"]
                result["BDA_pct"] = composition_row["BDA_pct"]
                result["FA_pct"] = composition_row["FA_pct"]
            rows_out.append(result)

    summary_df = pd.DataFrame(rows_out)

    summary_csv = os.path.join(output_dir, "KPFM_light_dark_distribution_fit_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    print("Saved individual KPFM distribution plots to:")
    print(dist_dir)
    print("Saved KPFM distribution fit summary:")
    print(summary_csv)

    return summary_df


def fit_single_spectrum_residual_aware(wl, y, well=None):
    wl_plot, y_corr_plot, y_smooth_plot, wl_fit, y_fit, y_smooth_fit = preprocess_spectrum(wl, y)

    if len(wl_fit) < 10 or np.nanmax(y_fit) <= 0:
        return {
            "ok": False,
            "wl_plot": wl_plot,
            "y_corr_plot": y_corr_plot,
            "y_smooth_plot": y_smooth_plot,
            "wl_fit": wl_fit,
            "fit_y_plot": np.full_like(wl_plot, np.nan),
            "peaks": [],
        }

    if well is not None and well in COMPOSITION_PERCENT_MAP:
        pea, bda, fa = COMPOSITION_PERCENT_MAP[well]
        if FORCE_ALL_FA_FREE_WELLS_TO_N1 and fa == 0:
            return make_forced_n1_fit_result(
                wl_plot,
                y_corr_plot,
                y_smooth_plot,
                wl_fit,
                reason="FA_free_endpoint_forced_n1",
            )

    if well is not None and is_overshoot_low_n_endpoint(well, wl_plot, y_corr_plot):
        return make_forced_n1_fit_result(
            wl_plot,
            y_corr_plot,
            y_smooth_plot,
            wl_fit,
            reason="overshoot_forced_n1",
        )

    y_max = max(np.nanmax(y_fit), 1e-9)
    phase_list = allowed_phases_for_well(well) if well is not None else list(PHASE_ORDER)

    initial_candidates = build_initial_candidates(wl_fit, y_fit, y_smooth_fit, phase_list=phase_list)
    _, initial_peaks_raw = fit_candidates(wl_fit, y_fit, initial_candidates)
    initial_peaks = filter_components(initial_peaks_raw, y_max)

    residual_candidates = add_residual_candidates(
        wl_fit,
        y_fit,
        y_smooth_fit,
        initial_peaks,
        phase_list=phase_list,
    )

    all_candidates = initial_candidates + residual_candidates

    by_phase = {}
    for cand in all_candidates:
        ph = cand["phase"]
        if ph not in by_phase or cand["A0"] > by_phase[ph]["A0"]:
            by_phase[ph] = cand

    final_candidates = [by_phase[p] for p in phase_list if p in by_phase]

    _, final_peaks_raw = fit_candidates(wl_fit, y_fit, final_candidates)
    final_peaks = filter_components(final_peaks_raw, y_max)


    final_peaks = [p for p in final_peaks if p["phase"] in phase_list]

    if len(final_peaks) == 0 and FORCE_ONE_COMPONENT_IF_EMPTY and len(initial_candidates) > 0:
        best = max(initial_candidates, key=lambda d: d["A0"])

        final_peaks = [{
            "phase": best["phase"],
            "source": "forced",
            "A": best["A0"],
            "mu_nm": best["mu0"],
            "sigma_nm": best["sigma0"],
            "area": area_of_gaussian(best["A0"], best["sigma0"]),
        }]

    if len(final_peaks) == 0:
        return {
            "ok": False,
            "wl_plot": wl_plot,
            "y_corr_plot": y_corr_plot,
            "y_smooth_plot": y_smooth_plot,
            "wl_fit": wl_fit,
            "fit_y_plot": np.full_like(wl_plot, np.nan),
            "peaks": [],
        }

    fit_params = []
    for p in final_peaks:
        fit_params.extend([p["A"], p["mu_nm"], p["sigma_nm"]])

    fit_y_plot = multi_gaussian(wl_plot, *fit_params)

    return {
        "ok": True,
        "wl_plot": wl_plot,
        "y_corr_plot": y_corr_plot,
        "y_smooth_plot": y_smooth_plot,
        "wl_fit": wl_fit,
        "fit_y_plot": fit_y_plot,
        "peaks": final_peaks,
    }


def summarize_fit_result(well, fit_result):
    out = {"Well": well}

    for phase in PHASE_ORDER:
        out[f"frac_{phase}"] = 0.0

    out["frac_Unassigned"] = 0.0

    peaks = fit_result["peaks"]
    out["num_peaks"] = len(peaks)

    if len(peaks) == 0:
        out["dominant_peak_nm"] = np.nan
        out["dominant_phase"] = np.nan
        out["total_fitted_area"] = np.nan
        return out

    total_area = sum(p["area"] for p in peaks)
    out["total_fitted_area"] = total_area

    phase_area = {phase: 0.0 for phase in PHASE_ORDER}
    phase_area["Unassigned"] = 0.0

    for p in peaks:
        ph = p["phase"]
        if ph not in phase_area:
            ph = "Unassigned"
        phase_area[ph] += p["area"]

    if total_area > 0:
        for phase, area in phase_area.items():
            out[f"frac_{phase}"] = area / total_area

    dominant_peak = max(peaks, key=lambda d: d["area"])

    out["dominant_peak_nm"] = dominant_peak["mu_nm"]
    out["dominant_phase"] = dominant_peak["phase"]

    return out


def make_peak_table_rows(well, fit_result):
    rows = []

    for i, p in enumerate(fit_result["peaks"], start=1):
        rows.append({
            "Well": well,
            "peak_number": i,
            "mu_nm": p["mu_nm"],
            "sigma_nm": p["sigma_nm"],
            "amplitude": p["A"],
            "area": p["area"],
            "phase": p["phase"],
            "source": p.get("source", "initial"),
        })

    return rows


def plot_fit_examples(matched_wells, fit_map, output_png, output_svg, title_prefix):
    if len(matched_wells) == 0:
        return []

    png_stem, png_extension = os.path.splitext(output_png)
    svg_stem, svg_extension = os.path.splitext(output_svg)
    outputs = []

    for well in matched_wells:
        fr = fit_map[well]
        fig, ax = plt.subplots(figsize=(5.0, 3.3), dpi=250)
        ax.plot(
            fr["wl_plot"],
            fr["y_corr_plot"],
            linewidth=1.2,
            label="PL baseline-corrected",
        )

        if fr["ok"]:
            ax.plot(
                fr["wl_plot"],
                fr["fit_y_plot"],
                "--",
                linewidth=1.4,
                label="endpoint-calibrated fit",
            )

            for peak in fr["peaks"]:
                component = gaussian1(
                    fr["wl_plot"],
                    peak["A"],
                    peak["mu_nm"],
                    peak["sigma_nm"],
                )
                ax.plot(fr["wl_plot"], component, ":", linewidth=1.0)
                ax.axvline(peak["mu_nm"], linestyle=":", linewidth=0.8)
                label = peak["phase"]
                if peak.get("source") == "residual":
                    label += " *"
                ax.text(
                    peak["mu_nm"],
                    np.nanmax(fr["y_corr_plot"]) * 0.92,
                    label,
                    rotation=90,
                    fontsize=7,
                    ha="center",
                    va="top",
                )

        pea, bda, fa = COMPOSITION_PERCENT_MAP[well]
        ax.set_title(
            f"{title_prefix}\n{well}: {pea}% PEA / {bda}% BDA / {fa}% FA\n"
            "* = component added from residual shoulder check",
            fontsize=9,
        )
        ax.set_xlim(PL_PLOT_WL_MIN, PL_PLOT_WL_MAX)
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Intensity (a.u.)")
        ax.grid(alpha=0.2)
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            ax.legend(handles, labels, loc="upper right", fontsize=7)
        fig.tight_layout()
        well_png = f"{png_stem}_{well}{png_extension}"
        well_svg = f"{svg_stem}_{well}{svg_extension}"
        fig.savefig(well_png, dpi=500, bbox_inches="tight")
        fig.savefig(well_svg, bbox_inches="tight")
        plt.close(fig)
        outputs.extend([well_png, well_svg])

    return outputs


def run_phasefit_for_timepoint(
    timepoint,
    time_min,
    time_label,
    blocks,
    composition,
    kpfm_avg,
    pv_col,
    pv_label,
    output_prefix,
    make_fit_examples=True,
):
    chosen_blocks = blocks[PL_READ_FOR_ANALYSIS]

    chosen_block = chosen_blocks[timepoint - 1].copy()
    wl_all = pd.to_numeric(chosen_block["Wavelength"], errors="coerce").to_numpy()

    spectra_map = {}

    for well in WELLS_ROW_MAJOR:
        y = pd.to_numeric(chosen_block[well], errors="coerce").to_numpy()
        spectra_map[well] = (wl_all.copy(), y.copy())

    matched = composition.merge(
        kpfm_avg,
        on=["Well", "KPFM_index"],
        how="inner",
    )

    matched = sort_matched_df(matched, SORT_MODE, pv_col=pv_col).reset_index(drop=True)

    print(f"\nMatched KPFM wells for {time_label}:")
    display(matched[["Well", "PEA_pct", "BDA_pct", "FA_pct", pv_col]])

    fit_summaries = []
    peak_rows = []
    fit_map = {}

    for _, row in matched.iterrows():
        well = row["Well"]
        wl, y = spectra_map[well]

        fr = fit_single_spectrum_residual_aware(wl, y, well=well)
        fit_map[well] = fr

        fit_summaries.append(summarize_fit_result(well, fr))
        peak_rows.extend(make_peak_table_rows(well, fr))

    fit_summary_df = pd.DataFrame(fit_summaries)
    peak_table_df = pd.DataFrame(peak_rows)

    fit_summary_csv = os.path.join(OUTPUT_DIR, f"{output_prefix}_endpoint_calibrated_phase_summary_by_well.csv")
    peak_table_csv = os.path.join(OUTPUT_DIR, f"{output_prefix}_endpoint_calibrated_peak_table_by_well.csv")

    fit_summary_df.to_csv(fit_summary_csv, index=False)
    peak_table_df.to_csv(peak_table_csv, index=False)

    plot_df = matched.merge(fit_summary_df, on="Well", how="left")
    plot_df = sort_matched_df(plot_df, SORT_MODE, pv_col=pv_col).reset_index(drop=True)
    plot_df["Composition_label"] = plot_df.apply(make_composition_label, axis=1)

    plot_csv = os.path.join(OUTPUT_DIR, f"{output_prefix}_endpoint_calibrated_plotting_table_phasefit_vs_KPFM.csv")
    plot_df.to_csv(plot_csv, index=False)

    print(f"\nFinal plotting table for {time_label}:")
    display(plot_df[[
        "Well",
        "PEA_pct",
        "BDA_pct",
        "FA_pct",
        pv_col,
        "num_peaks",
        "dominant_peak_nm",
        "dominant_phase",
    ] + [f"frac_{p}" for p in PHASE_ORDER]])

    if plot_df["dominant_peak_nm"].notna().sum() >= 3:
        corr_dom = plot_df.dropna(subset=["dominant_peak_nm", pv_col]).copy()

        pear_dom = pearsonr(corr_dom["dominant_peak_nm"], corr_dom[pv_col])
        spear_dom = spearmanr(corr_dom["dominant_peak_nm"], corr_dom[pv_col])

        corr_dom_text = (
            f"Dominant peak vs KPFM:\n"
            f"Pearson r = {pear_dom.statistic:.2f}, p = {pear_dom.pvalue:.2g}\n"
            f"Spearman ρ = {spear_dom.statistic:.2f}, p = {spear_dom.pvalue:.2g}\n"
            f"n = {len(corr_dom)}"
        )
    else:
        corr_dom_text = "Not enough points for dominant peak vs KPFM correlation"

    print("\n" + corr_dom_text)

    x = np.arange(len(plot_df))


    figure_width = max(12, 0.85 * len(plot_df) + 4)

    fig_phase, ax_phase = plt.subplots(figsize=(figure_width, 5.2), dpi=300)
    bottom = np.zeros(len(plot_df), dtype=float)
    for phase in PHASE_ORDER:
        vals = plot_df[f"frac_{phase}"].fillna(0).to_numpy()
        ax_phase.bar(
            x,
            vals,
            bottom=bottom,
            width=0.78,
            label=phase,
            color=PHASE_COLORS[phase],
            edgecolor="black",
            linewidth=0.3,
        )
        bottom += vals
    ax_phase.set_ylabel("PL-derived emissive component fraction")
    ax_phase.set_xlabel("Composition")
    ax_phase.set_ylim(0, 1.02)
    ax_phase.set_title(
        "Endpoint-calibrated PL-derived emissive component distribution\n"
        f"{PL_READ_FOR_ANALYSIS} PL, {time_label}, {time_min} min"
    )
    ax_phase.set_xticks(x)
    ax_phase.set_xticklabels(plot_df["Composition_label"], fontsize=8)
    ax_phase.legend(loc="upper left", ncol=3, fontsize=9)
    ax_phase.grid(axis="y", alpha=0.25)
    for i, row in plot_df.iterrows():
        if pd.notna(row["dominant_phase"]):
            ax_phase.text(
                i,
                1.01,
                str(row["dominant_phase"]),
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=90,
            )
    phase_png = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_ENDPOINT_CALIBRATED_PHASE_DISTRIBUTION.png",
    )
    phase_svg = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_ENDPOINT_CALIBRATED_PHASE_DISTRIBUTION.svg",
    )
    fig_phase.savefig(phase_png, dpi=600, bbox_inches="tight")
    fig_phase.savefig(phase_svg, bbox_inches="tight")
    plt.close(fig_phase)

    fig_kpfm, ax_kpfm = plt.subplots(figsize=(figure_width, 4.2), dpi=300)
    ax_kpfm.plot(
        x,
        plot_df[pv_col].to_numpy(),
        marker="o",
        linewidth=1.5,
        markersize=5,
        color="black",
    )
    ax_kpfm.set_ylabel(pv_label)
    ax_kpfm.set_xlabel("Composition")
    ax_kpfm.set_xticks(x)
    ax_kpfm.set_xticklabels(plot_df["Composition_label"], fontsize=8)
    ax_kpfm.set_title(f"{pv_label}; {time_label}, {time_min} min")
    ax_kpfm.grid(alpha=0.25)
    for i, row in plot_df.iterrows():
        ax_kpfm.text(
            i,
            row[pv_col],
            row["Well"],
            fontsize=7,
            ha="center",
            va="bottom",
        )
    kpfm_png = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_KPFM_RESPONSE.png",
    )
    kpfm_svg = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_KPFM_RESPONSE.svg",
    )
    fig_kpfm.savefig(kpfm_png, dpi=600, bbox_inches="tight")
    fig_kpfm.savefig(kpfm_svg, bbox_inches="tight")
    plt.close(fig_kpfm)

    phase_color_list = []
    for dominant_phase in plot_df["dominant_phase"]:
        if pd.isna(dominant_phase):
            phase_color_list.append("#808080")
        else:
            phase_color_list.append(PHASE_COLORS.get(dominant_phase, "#808080"))
    sizes = 60 + 35 * plot_df["num_peaks"].fillna(0).to_numpy()
    fig_peak, ax_peak = plt.subplots(figsize=(figure_width, 4.8), dpi=300)
    ax_peak.scatter(
        x,
        plot_df["dominant_peak_nm"].to_numpy(),
        s=sizes,
        c=phase_color_list,
        edgecolors="black",
        linewidths=0.4,
    )
    for i, row in plot_df.iterrows():
        if pd.notna(row["dominant_peak_nm"]):
            ax_peak.text(
                i,
                row["dominant_peak_nm"] + 8,
                f"{int(row['num_peaks'])} comp.",
                fontsize=7,
                ha="center",
                va="bottom",
            )
    for phase in PHASE_ORDER:
        lo, hi = PHASE_WINDOWS[phase]
        ax_peak.axhspan(lo, hi, color=PHASE_COLORS[phase], alpha=0.06)
    ax_peak.set_ylabel("Dominant fitted peak (nm)")
    ax_peak.set_xlabel("Composition")
    ax_peak.set_xticks(x)
    ax_peak.set_xticklabels(plot_df["Composition_label"], fontsize=8)
    ax_peak.set_title(f"Dominant fitted PL peak; {time_label}, {time_min} min")
    ax_peak.grid(alpha=0.25)
    ax_peak.text(
        0.99,
        0.98,
        corr_dom_text,
        transform=ax_peak.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
    )
    peak_png = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_DOMINANT_PL_PEAK.png",
    )
    peak_svg = os.path.join(
        OUTPUT_DIR,
        f"{output_prefix}_DOMINANT_PL_PEAK.svg",
    )
    fig_peak.savefig(peak_png, dpi=600, bbox_inches="tight")
    fig_peak.savefig(peak_svg, bbox_inches="tight")
    plt.close(fig_peak)

    print("Saved standalone phase, KPFM, and dominant-peak figures:")
    print(phase_png)
    print(phase_svg)
    print(kpfm_png)
    print(kpfm_svg)
    print(peak_png)
    print(peak_svg)


    compact_df = plot_df.copy()
    compact_df["KPFM_norm_to_1"] = normalize_overlay_to_0_1(compact_df[pv_col], mode="auto")

    fig2, axc = plt.subplots(
        figsize=(max(12, 0.85 * len(compact_df) + 4), 5.2),
        dpi=300,
    )

    bottom = np.zeros(len(compact_df), dtype=float)

    for phase in PHASE_ORDER:
        vals = compact_df[f"frac_{phase}"].fillna(0).to_numpy()

        axc.bar(
            x,
            vals,
            bottom=bottom,
            width=0.78,
            color=PHASE_COLORS[phase],
            edgecolor="black",
            linewidth=0.3,
            label=phase,
        )

        bottom += vals

    axc2 = axc.twinx()

    axc2.plot(
        x,
        compact_df["KPFM_norm_to_1"].to_numpy(),
        color="black",
        marker="o",
        linewidth=1.8,
        markersize=5,
    )

    axc2.set_ylim(0, 1.05)
    axc2.set_ylabel(f"Normalized {pv_label}")

    axc.set_ylim(0, 1.02)
    axc.set_ylabel("PL-derived emissive component fraction")

    axc.set_xticks(x)
    axc.set_xticklabels(
        compact_df["Composition_label"],
        fontsize=8,
    )

    axc.set_title(
        f"Endpoint-calibrated PL-derived emissive component distribution with normalized {pv_label} overlay\n"
        f"{time_label}, {time_min} min"
    )

    axc.grid(axis="y", alpha=0.25)
    axc.legend(loc="upper left", ncol=3, fontsize=8)

    compact_png = os.path.join(OUTPUT_DIR, f"{output_prefix}_COMPACT_endpoint_calibrated_phase_plus_KPFM.png")
    compact_svg = os.path.join(OUTPUT_DIR, f"{output_prefix}_COMPACT_endpoint_calibrated_phase_plus_KPFM.svg")

    fig2.savefig(compact_png, dpi=600, bbox_inches="tight")
    fig2.savefig(compact_svg, bbox_inches="tight")

    plt.show()

    print("Saved compact figure:")
    print(compact_png)
    print(compact_svg)


    if make_fit_examples:
        fit_examples_png = os.path.join(OUTPUT_DIR, f"{output_prefix}_ENDPOINT_CALIBRATED_PL_fit_examples.png")
        fit_examples_svg = os.path.join(OUTPUT_DIR, f"{output_prefix}_ENDPOINT_CALIBRATED_PL_fit_examples.svg")

        plot_fit_examples(
            matched_wells=plot_df["Well"].tolist(),
            fit_map=fit_map,
            output_png=fit_examples_png,
            output_svg=fit_examples_svg,
            title_prefix=f"Endpoint-calibrated PL fits: {time_label}, {time_min} min",
        )

        print("Saved fit example figure:")
        print(fit_examples_png)
        print(fit_examples_svg)

    return plot_df


composition = make_composition_df()

print("\nLoading and parsing PL data...")
blocks = parse_plate_reader_blocks(CSV_PATH)

pl_top = summarize_pl(blocks["top"], "top")
pl_bottom = summarize_pl(blocks["bottom"], "bottom")

pl_all = pd.concat([pl_top, pl_bottom], ignore_index=True)

if pl_all.empty:
    raise ValueError("PL parsing produced no data.")

primary_timepoint, primary_time_min, primary_label, valid_counts = choose_timepoint(
    pl_all=pl_all,
    mode=PRIMARY_PL_TIMEPOINT_MODE,
    read_name=PL_READ_FOR_ANALYSIS,
)

last_timepoint, last_time_min, last_label, _ = choose_timepoint(
    pl_all=pl_all,
    mode="last_valid",
    read_name=PL_READ_FOR_ANALYSIS,
)

print("\nLoading and parsing photoKPFM data...")
kpfm_df = summarize_kpfm(H5_PATH)


if MAKE_KPFM_DISTRIBUTION_PLOTS:
    print("\nMaking actual light/dark photoKPFM surface-potential distribution plots...")

    kpfm_distribution_summary = pd.DataFrame()

    if MAKE_ALL_KPFM_DISTRIBUTION_PLOTS:
        kpfm_distribution_summary = make_all_kpfm_distribution_plots(
            h5_path=H5_PATH,
            composition_df=composition,
            output_dir=OUTPUT_DIR,
            crop=KPFM_DISTRIBUTION_CROP,
            bins=KPFM_DISTRIBUTION_BINS,
            hist_range=KPFM_DISTRIBUTION_HIST_RANGE,
        )

        if not kpfm_distribution_summary.empty:
            print("\nKPFM distribution fit summary:")
            display(kpfm_distribution_summary)

pv_col = "photovoltage_abs_V" if USE_ABS_PHOTOVOLTAGE else "photovoltage_signed_V"
pv_label = "|photoKPFM ΔV| (V)" if USE_ABS_PHOTOVOLTAGE else "photoKPFM ΔV (V)"

kpfm_avg = (
    kpfm_df
    .groupby("Well", as_index=False)
    .agg({
        "KPFM_index": "first",
        "photovoltage_signed_V": "mean",
        "photovoltage_abs_V": "mean",
        "mu_dark_V": "mean",
        "mu_light_V": "mean",
    })
)

print("Unique KPFM-measured wells:", len(kpfm_avg))
display(kpfm_avg)


kpfm_overlay_specs = [
    {
        "col": pv_col,
        "label": pv_label,
        "prefix": "PHOTOVOLTAGE_DELTA",
        "make_fit_examples": True,
    }
]

if MAKE_KPFM_DELTA_LIGHT_DARK_FIGURES:
    kpfm_overlay_specs.extend([
        {
            "col": "mu_light_V",
            "label": "photoKPFM light surface potential, μ_light (V)",
            "prefix": "LIGHT_SURFACE_POTENTIAL",
            "make_fit_examples": False,
        },
        {
            "col": "mu_dark_V",
            "label": "photoKPFM dark surface potential, μ_dark (V)",
            "prefix": "DARK_SURFACE_POTENTIAL",
            "make_fit_examples": False,
        },
    ])

primary_plot_dfs = {}

for spec in kpfm_overlay_specs:
    if spec["col"] not in kpfm_avg.columns:
        print(f"Skipping {spec['prefix']} because {spec['col']} was not found.")
        continue

    if kpfm_avg[spec["col"]].notna().sum() == 0:
        print(f"Skipping {spec['prefix']} because {spec['col']} is all NaN.")
        continue

    primary_plot_dfs[spec["prefix"]] = run_phasefit_for_timepoint(
        timepoint=primary_timepoint,
        time_min=primary_time_min,
        time_label=f"initial timepoint {primary_timepoint}",
        blocks=blocks,
        composition=composition,
        kpfm_avg=kpfm_avg,
        pv_col=spec["col"],
        pv_label=spec["label"],
        output_prefix=f"PRIMARY_INITIAL_{spec['prefix']}",
        make_fit_examples=spec["make_fit_examples"],
    )

if MAKE_LAST_VALID_SENSITIVITY_FIGURE and last_timepoint != primary_timepoint:
    sensitivity_plot_dfs = {}

    for spec in kpfm_overlay_specs:
        if spec["col"] not in kpfm_avg.columns:
            continue

        if kpfm_avg[spec["col"]].notna().sum() == 0:
            continue

        sensitivity_plot_dfs[spec["prefix"]] = run_phasefit_for_timepoint(
            timepoint=last_timepoint,
            time_min=last_time_min,
            time_label=f"last valid timepoint {last_timepoint}",
            blocks=blocks,
            composition=composition,
            kpfm_avg=kpfm_avg,
            pv_col=spec["col"],
            pv_label=spec["label"],
            output_prefix=f"SENSITIVITY_LAST_VALID_{spec['prefix']}",
            make_fit_examples=False,
        )

elif MAKE_LAST_VALID_SENSITIVITY_FIGURE:
    print("Initial and last valid timepoints are the same, so no separate sensitivity figure was made.")


zip_path = "endpoint_calibrated_phasefit_PL_KPFM_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_in_dir in os.walk(OUTPUT_DIR):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, OUTPUT_DIR)
            zf.write(full_path, arcname)

print("\nAll done.")
print("Created:", zip_path)
print("\nStandalone figure suffixes:")
print("ENDPOINT_CALIBRATED_PHASE_DISTRIBUTION")
print("KPFM_RESPONSE")
print("DOMINANT_PL_PEAK")
print("PRIMARY_INITIAL_PHOTOVOLTAGE_DELTA_COMPACT_endpoint_calibrated_phase_plus_KPFM.png")
print("PRIMARY_INITIAL_LIGHT_SURFACE_POTENTIAL_COMPACT_endpoint_calibrated_phase_plus_KPFM.png")
print("PRIMARY_INITIAL_DARK_SURFACE_POTENTIAL_COMPACT_endpoint_calibrated_phase_plus_KPFM.png")
print("\nFit-checking figure:")
print("PRIMARY_INITIAL_PHOTOVOLTAGE_DELTA_ENDPOINT_CALIBRATED_PL_fit_examples.png")
print("\nSensitivity figures:")
print("Sensitivity runs use the same standalone figure suffixes.")
print("\nPrimary PL timepoint:", primary_timepoint, "at", primary_time_min, "min")
print("Last valid PL timepoint:", last_timepoint, "at", last_time_min, "min")

print("\nKPFM light/dark pixel-distribution outputs:")
print("KPFM_light_dark_distribution_plots/  (individual well histograms)")
print("KPFM_light_dark_distribution_fit_summary.csv")

files.download(zip_path)
